In [2]:
!pip install pandas --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd

file_path = '../data/raw_data.csv'

kolom_penting = [
    'Location_Identifier',
    'Activity_StartDate',
    'Activity_StartTime',
    'Result_Characteristic',
    'Result_Measure',
    'Result_MeasureUnit'
]

df = pd.read_csv(file_path, usecols=kolom_penting, low_memory=False)

df.head()

  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/1f/67/af63f83cd6ca603a00fe8530c10a60f0879265b8be00b5930e8e78c5b30b/pandas-3.0.1-cp311-cp311-win_amd64.whl.metadata
  Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Obtaining dependency information for numpy>=1.26.0 from https://files.pythonhosted.org/packages/76/ae/e0265e0163cf127c24c3969d29f1c4c64551a1e375d95a13d32eab25d364/numpy-2.4.2-cp311-cp311-win_amd64.whl.metadata
  Using cached numpy-2.4.2-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Obtaining dependency information for tzdata from https://files.pythonhosted.org/packages/c7/b0/003792df09decd6849a5e39c28b513c06e84436a54440380862b5aeff25d/tzdata-2025.3-py2.py3-none-any.whl.metadata
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.2-cp311-cp311-win_amd64.whl (12.6 MB)
Using cached tzdata-2025.3-py2.py3-none-any.whl 


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,Location_Identifier,Activity_StartDate,Activity_StartTime,Result_Characteristic,Result_Measure,Result_MeasureUnit
0,USGS-11303500,2021-09-02,17:15:00,"Temperature, water",23.2,deg C
1,USGS-11447650,2021-10-29,10:02:00,"Temperature, water",15.9,deg C
2,USGS-11447650,2020-05-01,10:51:00,"Temperature, water",21.2,deg C
3,USGS-11172175,2021-12-20,14:28:00,"Temperature, water",9.6,deg C
4,USGS-390016121015901,2020-11-18,07:40:00,Specific conductance,57.0,uS/cm


In [4]:
df['Result_Characteristic'].value_counts()

Result_Characteristic
Temperature, water       6690
Specific conductance     2338
pH                       1581
Dissolved oxygen (DO)    1394
Name: count, dtype: int64

In [ ]:
df = df.dropna(subset=['Result_Measure'])

df['Result_MeasureUnit'] = df['Result_MeasureUnit'].fillna('').astype(str).str.strip()

mask_f = (df['Result_Characteristic'] == 'Temperature, water') & (df['Result_MeasureUnit'] == 'deg F')
df.loc[mask_f, 'Result_Measure'] = (df.loc[mask_f, 'Result_Measure'] - 32) * 5.0 / 9.0
df.loc[mask_f, 'Result_MeasureUnit'] = 'deg C'

df['Parameter_with_Unit'] = df['Result_Characteristic'] + " (" + df['Result_MeasureUnit'] + ")"

df_pivot = pd.pivot_table(
    df, 
    values='Result_Measure', 
    index=['Location_Identifier', 'Activity_StartDate', 'Activity_StartTime'],
    columns='Parameter_with_Unit',
    aggfunc='mean'
).reset_index()

df_pivot.columns.name = None

df_pivot['Activity_StartDate'] = pd.to_datetime(df_pivot['Activity_StartDate'])

df_pivot = df_pivot.sort_values(by=['Location_Identifier', 'Activity_StartDate', 'Activity_StartTime'])

df_pivot = df_pivot.reset_index(drop=True)

df_pivot.sample(10)

,Location_Identifier,Activity_StartDate,Activity_StartTime,Dissolved oxygen (DO) (mg/L),Specific conductance (uS/cm),"Temperature, water (deg C)",pH (standard units)
3356,USGS-11303500,2020-10-06,07:15:00,NaN,NaN,20.7,NaN
1168,USGS-11074000,2021-09-02,14:20:00,9.7,1120.0,24.4,8.2
4170,USGS-11336790,2021-03-30,10:45:00,9.9,206.0,14.9,7.9
467,USGS-10336660,2021-02-04,10:30:00,11.3,79.0,0.5,NaN
2281,USGS-11176145,2021-11-09,14:08:00,NaN,NaN,13.7,NaN
6459,USGS-382113121383501,2021-08-31,11:06:00,7.7,234.0,23.1,8.2
4416,USGS-11447650,2020-08-27,07:44:00,NaN,NaN,21.5,NaN
4424,USGS-11447650,2020-09-02,11:20:00,NaN,NaN,22.1,NaN
4014,USGS-11303500,2022-01-27,16:24:00,NaN,NaN,11.4,NaN
3082,USGS-11303500,2020-01-06,10:45:00,NaN,NaN,10.3,NaN
